In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# Change this path if necessary
PROJECT_DIR = Path(r"D:\Big Data Project")

DATA_DIR = PROJECT_DIR / "data" / "raw"

print("Dataset directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

Dataset directory: D:\Big Data Project\data\raw
Directory exists: True


## Verify dataset file sizes

In [4]:
files = sorted(DATA_DIR.glob("*.csv"))

file_info = []

for file in files:
    size_mb = file.stat().st_size / (1024**2)

    file_info.append({
        "File": file.name,
        "Size (MB)": round(size_mb, 2)
    })

file_df = pd.DataFrame(file_info)

display(file_df)

print(
    "Total size (GB):",
    round(file_df["Size (MB)"].sum() / 1024, 2)
)

,File,Size (MB)
0,building_metadata.csv,0.04
1,sample_submission.csv,426.83
2,test.csv,1394.71
3,train.csv,647.18
4,weather_test.csv,14.10
5,weather_train.csv,7.10


Total size (GB): 2.43


## Inspect the energy readings

In [5]:
train_path = DATA_DIR / "train.csv"

train_sample = pd.read_csv(
    train_path,
    nrows=10
)

display(train_sample)

print("Column names:")
print(train_sample.columns.tolist())

print("\nData types:")
print(train_sample.dtypes)

,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01 00:00:00,0.0
1,1,0,2016-01-01 00:00:00,0.0
2,2,0,2016-01-01 00:00:00,0.0
3,3,0,2016-01-01 00:00:00,0.0
4,4,0,2016-01-01 00:00:00,0.0
5,5,0,2016-01-01 00:00:00,0.0
6,6,0,2016-01-01 00:00:00,0.0
7,7,0,2016-01-01 00:00:00,0.0
8,8,0,2016-01-01 00:00:00,0.0
9,9,0,2016-01-01 00:00:00,0.0


Column names:
['building_id', 'meter', 'timestamp', 'meter_reading']

Data types:
building_id        int64
meter              int64
timestamp            str
meter_reading    float64
dtype: object


## Inspect building characteristics

In [6]:
building_path = DATA_DIR / "building_metadata.csv"

buildings = pd.read_csv(building_path)

print("Shape:", buildings.shape)

display(buildings.head(10))

print("\nMissing values:")
print(buildings.isna().sum())

print("\nBuilding categories:")
print(buildings["primary_use"].value_counts())

Shape: (1449, 6)


,site_id,building_id,primary_use,square_feet,year_built,floor_count
0,0,0,Education,7432,2008.0,NaN
1,0,1,Education,2720,2004.0,NaN
2,0,2,Education,5376,1991.0,NaN
3,0,3,Education,23685,2002.0,NaN
4,0,4,Education,116607,1975.0,NaN
5,0,5,Education,8000,2000.0,NaN
6,0,6,Lodging/residential,27926,1981.0,NaN
7,0,7,Education,121074,1989.0,NaN
8,0,8,Education,60809,2003.0,NaN
9,0,9,Office,27000,2010.0,NaN



Missing values:
site_id           0
building_id       0
primary_use       0
square_feet       0
year_built      774
floor_count    1094
dtype: int64

Building categories:
primary_use
Education                        549
Office                           279
Entertainment/public assembly    184
Public services                  156
Lodging/residential              147
Other                             25
Healthcare                        23
Parking                           22
Warehouse/storage                 13
Manufacturing/industrial          12
Retail                            11
Services                          10
Technology/science                 6
Food sales and service             5
Utility                            4
Religious worship                  3
Name: count, dtype: int64


## Inspect weather observations

In [7]:
weather_path = DATA_DIR / "weather_train.csv"

weather_sample = pd.read_csv(
    weather_path,
    nrows=10
)

display(weather_sample)

print("\nColumn names:")
print(weather_sample.columns.tolist())

print("\nData types:")
print(weather_sample.dtypes)

,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2016-01-01 00:00:00,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,0,2016-01-01 01:00:00,24.4,NaN,21.1,-1.0,1020.2,70.0,1.5
2,0,2016-01-01 02:00:00,22.8,2.0,21.1,0.0,1020.2,0.0,0.0
3,0,2016-01-01 03:00:00,21.1,2.0,20.6,0.0,1020.1,0.0,0.0
4,0,2016-01-01 04:00:00,20.0,2.0,20.0,-1.0,1020.0,250.0,2.6
5,0,2016-01-01 05:00:00,19.4,NaN,19.4,0.0,NaN,0.0,0.0
6,0,2016-01-01 06:00:00,21.1,6.0,21.1,-1.0,1019.4,0.0,0.0
7,0,2016-01-01 07:00:00,21.1,NaN,21.1,0.0,1018.8,210.0,1.5
8,0,2016-01-01 08:00:00,20.6,NaN,20.0,0.0,1018.1,0.0,0.0
9,0,2016-01-01 09:00:00,21.1,NaN,20.6,0.0,1019.0,290.0,1.5



Column names:
['site_id', 'timestamp', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed']

Data types:
site_id                 int64
timestamp                 str
air_temperature       float64
cloud_coverage        float64
dew_temperature       float64
precip_depth_1_hr     float64
sea_level_pressure    float64
wind_direction        float64
wind_speed            float64
dtype: object


## Verify the dataset without loading it into RAM

In [8]:
import duckdb

In [9]:
con = duckdb.connect()

train_file = train_path.as_posix()

query = """
SELECT
    meter,
    COUNT(*) AS total_records,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    MIN(meter_reading) AS minimum_reading,
    MAX(meter_reading) AS maximum_reading
FROM read_csv_auto(?, header=true)
GROUP BY meter
ORDER BY meter
"""

meter_summary = con.execute(
    query,
    [train_file]
).df()

display(meter_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,meter,total_records,first_timestamp,last_timestamp,minimum_reading,maximum_reading
0,0,12060910,2016-01-01,2016-12-31 23:00:00,0.0,79769.0
1,1,4182440,2016-01-01,2016-12-31 23:00:00,0.0,880374.0
2,2,2708713,2016-01-01,2016-12-31 23:00:00,0.0,21904700.0
3,3,1264037,2016-01-01,2016-12-31 23:00:00,0.0,160187.0
